# How the Dataset is Represented

The dataset is stored as dictionary with other dictionaries in it. 

- dictionary
    - problem #x (type dictionary | indicates we can have x amount of problems)
        - axiom (type str)
        - conjecture (type str)
        - cnfrefutation (type dictionary)
            - refutation step #y (type dictionary)
                - language (type str)
                - name (type str)
                - type (type str)
                - logical formula (type str)
                - source (type dictionary | some refutation steps may not contain this)
                    - inference #z (type dictionary)
                        - name (type str)
                        - type (type str)
                        - name of premises (type str)
                - optional useful parameter (some refutation steps may not contain this)

In [192]:
import io
import subprocess
import pickle
import copy
import gzip
import tqdm
import re
from itertools import islice
import json

In [193]:
gzip_location = './external/gzip/new/theorum.pickle.gzip' #gzip file
txt_location = "./external/txt/theorem.txt" #txt file of a modified version of gzip_location
parsed_file_to_txt = "./external/txt/parsed data/theorem_parsed.txt" #Location of parsed txt file
parsed_file_to_json = "./external/txt/parsed data/theorem_parsed.json" #Location of parsed txt file

In [194]:
with gzip.open(gzip_location,'rb') as f:
    theorem_gzip = pickle.load(f)

Displaying Dataset

In [195]:
print("Rows:", len(theorem_gzip))
print("Columns:", len(theorem_gzip[0]))
print(theorem_gzip[0][0])
print(theorem_gzip[0][1])
print(theorem_gzip[0][2])
print(theorem_gzip[0][3])

Rows: 10000
Columns: 4
(b)&(~b|~c) entails (~a|b)&(b|c).
fof('axio(b)&(~b|~c)',axiom,(b)&(~b|~c)). fof('conj(~a|b)&(b|c)',conjecture,(~a|b)&(b|c)).
Found
CompletedProcess(args=['/home/anmarch/source/eprover/PROVER/eprover', '--proof-object', 'f.tptp'], returncode=0, stdout=b"# Initializing proof state\n# Scanning for AC axioms\n#\n#cnf(i_0_2, plain, (b)).\n#\n#cnf(i_0_6, negated_conjecture, (a)).\n##\n# Proof found!\n# SZS status Theorem\n# SZS output start CNFRefutation\nfof('conj(~a|b)&(b|c)', conjecture, ((~(a)|b)&(b|c)), file('f.tptp', 'conj(~a|b)&(b|c)')).\nfof('axio(b)&(~b|~c)', axiom, (b&(~(b)|~(c))), file('f.tptp', 'axio(b)&(~b|~c)')).\nfof(c_0_2, negated_conjecture, ~(((~a|b)&(b|c))), inference(fof_simplification,[status(thm)],[inference(assume_negation,[status(cth)],['conj(~a|b)&(b|c)'])])).\nfof(c_0_3, negated_conjecture, (((~b|a)&(~c|a))&((~b|~b)&(~c|~b))), inference(distribute,[status(thm)],[inference(fof_nnf,[status(thm)],[inference(fof_nnf,[status(thm)],[c_0_2])])])).\nf

Write gzip to .txt file for parsing.

In [196]:
f = open(txt_location, "w")

#range is currently 5 to parse through 5 theorems of our dataset. The final version will use len(theorem_gzip).
for x in range(100): 
    for y in range(len(theorem_gzip[x])):      
        #Used to write the first line of our .txt file.
        if x == 0 and y == 0:
            f.write("theorem #" + str(x) + "\n" + str(theorem_gzip[x][y]).replace("\\n", ""))
        
        #Used to write the rest of the lines of our .txt file.
        elif x != 0 and y == 0:
            f.write("\ntheorem #" + str(x) + "\n" + str(theorem_gzip[x][y]).replace("\\n", ""))
        
        #Adds newline to string since theorum[x][1] doesn't have newlines.
        elif y == 1:
            f.write("\n" + str(theorem_gzip[x][y]).replace(" ", "\n"))
        
        #Case to consider for other columns.
        else:
            f.write("\n" + str(theorem_gzip[x][y]).replace("\\n", "\n"))
f.close()

Obtain each line of the .txt file for parsing.

In [197]:
lines = []

def fn_read_file(file):
    with open(file, "r") as f:
        return f.readlines()

lines = fn_read_file(txt_location)

Deleting lines we don't need. Examples:
* CompletedProces (args=['/home/user/...

In [198]:
with open(txt_location, "w") as f:
    for line in lines:
        #Problem: not able to remove the last string. 
        if ("CompletedProcess" not in line) and ("\n" != line) and ("##\n" != line) and ("#\n" != line) and ("\", stderr=b\'\')\n" != line):
            f.write(line)
lines = fn_read_file(txt_location)

<div class = "title">
    <h1>
        <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#formula_data">&ltFormula Data&gt</a> Parser
    </h1>
</div>

\<Formula Data\> can be in the form of the following:
 - <u>\<Formula Data\> ::= $thf(\<thf_formula\>) | $fof(\<fof_formula\>) | $cnf(\<cnf_formula\>) | $fot(\<term\>)</u>

Our datasets only ever contain fof and cnf. In this markdown cell, only fof is covered. 
- $fof(\<fof_formula\>) only ever contains 3 parameters in total. Some examples of each parameters are provided. 
    - <u><a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#fof_annotated">\<fof_annotated\></a> ::= fof\(<name\>,\<formula_role\>,\<fof_formula\>\<annotations\>)</u>
        - <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#name">\<name\></a> Examples are: 
            - '(a)'
            - '(a|b)'
            - '(c|b)&(a|~a)' (including the apostrophes). 
        - <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#formula_role">\<formula_role\></a> Examples are:
            - axiom
            - conjecture
            - negated_conjecture
            - plain
        - <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#fof_formula">\<fof_formula\></a> Examples are: 
            - (a)
            - (c)
            - ~((b|a))
            - ~(((c|b)&(a|~a)))
        - <u><a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#annotations">\<annotations\></a>:</u>
            - This particularly "adds" another parameter to \<fof_annotated\> if needed. It could also be empty. If \<annotations\> is used, a possible example is \<inference_record\>
            - <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#inference_record">\<inference_record\></a> ::= inference(\<inference_rule>, \<useful_info\>, [\<parent_list\>])
                - Examples for \<inference_record\> are:
                    - inference(fof_simplification,[status(thm)],[inference(assume_negation,[status(cth)],['(a)'])])
                    - inference(assume_negation,[status(cth)],['(a)'])
                - <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#inference_rule">\<inference_rule\></a> Examples are:
                    - fof_simplification
                - <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#useful_info">\<useful_info\></a> Examples are:
                    - [status(thm)]
                    - [status(cth)]
                - <a href = "https://tptp.org/Seminars/THF/SyntaxBNF.html#parent_list">\<parent_list\></a> Examples are:
                    - ['(a)' ]


<h1>Parsing dataset from theorem.txt</h1>

Obtaining the following:
- Axioms
- Conjectures
- Cnfrefutations
    - Cnfrefutations contain are the proof steps and are further parsed in other cells.

In [199]:
problems = {} #List of dictionaries. Each dictionary is a theorem with its respective axioms, conjectures, and cnfrefutation.

with open(txt_location, "r") as f:
    temp_dict = {}
    problem_id = 0
    for line in f: 
        if ("theorem #0" not in line) and ("theorem #" in line): 
            problems.update({f"problem #{problem_id}":temp_dict})  # Append the dictionary to the list of axioms
            #print(json.dumps(temp_dict, indent = 4, sort_keys = True)) #Prints out the respective theorem's information.
            problem_id += 1
            temp_dict = {}  # Reset the dictionary for each theorem
        #Grabs axioms
        if ("fof(\'" in line) and ("axiom" in line):
            temp_axiom = ""  
            for char in line[5:]:
                if char == "'":  # Stop collecting if reached closed parentheses
                    break  
                temp_axiom += char  # Append to temp_axiom
            temp_dict.update({"axiom": temp_axiom})
        
        #Grabs conjectures
        if ("fof(\'" in line) and ("conjecture" in line):
            temp_conject = ""
            temp_conject = ""  
            for char in line[5:]:
                if char == "'":  # Stop collecting if reached closed parentheses
                    break  
                temp_conject += char  # Append to temp_axiom
            temp_dict.update({"conjecture" : temp_conject})

        #Grabs AC axioms
        if "# Scanning for AC axioms" in line:
            temp_ac_axiom = ""
            while True:
                line = next(f, None)
                if "# Proof found!" in line:
                    break
                temp_ac_axiom = temp_ac_axiom + line.strip()
            temp_dict.update({"ac axiom" : temp_ac_axiom})

        #Grabs respective CNF Refutation and stores it in one string. Problem: There are multiple per theorem. 
        #Later section of the code resolves this issue.
        if "# SZS output start CNFRefutation" in line:
            temp_respective_cnfrefutation = ""
            while True:
                line = next(f, None)
                if line is None or "# SZS output end CNFRefutation" in line:
                    break  
                temp_respective_cnfrefutation = temp_respective_cnfrefutation + line.strip()
                #print(temp_respective_cnfrefutation)
            temp_dict.update({"cnfrefutation" : temp_respective_cnfrefutation})

    #Dictionary
    problems.update({f"problem #{problem_id}":temp_dict})  # Append the dictionary to the list of axioms
    #print(json.dumps(temp_dict, indent = 4, sort_keys = False)) #Prints out the respective theorem's information.
    temp_dict = {}  # Reset the dictionary for each theorem
f.close()

<h2>Parsing cnfrefutation and AC axiom</h2>



- A cnfref can contain the following:
    - Language (Examples are cnf, fof, tff, ...)
    - Name (arbitrary, but unique.)
        - '(a)'
        - c_0_1
    - Type 
        - Examples:
            - axiom
            - lemma
            - conjecture
    - Logical Formula 
        - Examples:
            - $false (indicates an empty clause)
            - (~a)
            - (b)
    - Source
        - Examples:
            - inference(sr, [status])
    - Optional Useful Information such as the conclusion of a proof.

<h3>Seperate each AC axiom in "ac axiom" key</h3>

In [200]:
ac_axioms = []
#for every problem 
for x in range(len(problems)):
    temp_ac_axioms = []
    temp_str = ""
    problems_ac_axioms_val = problems.get(f"problem #{x}").get("ac axiom") #Variable that grabs the value of the key "ac axioms" in the variable "problems"

    #Parses through every character in each problem's ac axioms
    for char in range(len(problems_ac_axioms_val)):
        #obtain string as we parse through each character
        temp_str = temp_str + problems_ac_axioms_val[char]
        #If we reach a period
        if (problems_ac_axioms_val[char] == "."):
            #We have reached at the end of a ac axioms step. Append it to temp_ac_axioms.
            temp_ac_axioms.append(temp_str)
            #print(temp_str)
            temp_str = "" #Reset string
    ac_axioms.append(temp_ac_axioms) #Append temp_ac_axioms to ac axioms

<h3>Update the "problems" dictionary accordingly</h3>

In [201]:
for x in range(len(ac_axioms)):
    temp_dict = {}
    for y in range(len(ac_axioms[x])):
        #print(saturation[x][y])
        #Adds each saturation step for a respective problem into a temp dictionary.
        temp_dict.update({f"ac axiom #{y}" : ac_axioms[x][y]})
    #print(json.dumps(temp_dict, indent = 4, sort_keys = False)) #Used for debugging. Displays that temp_dict holds respective steps for a problem.
    problems[f"problem #{x}"]["ac axiom"] = temp_dict

<h3>Seperate each refutation in the "refutation" key</h3>
Note that cnfrefutation is a variable that consists of multiple refutation steps stored in a straight line. It is not seperated.

In [202]:
cnfrefutation = []
#for every problem 
for x in range(len(problems)):
    temp_cnfref = []
    temp_str = ""
    problems_cnfref_val = problems.get(f"problem #{x}").get("cnfrefutation") #Variable that grabs the value of the key "cnfrefutation" in the variable "problems"

    #Parses through every character in each problem's cnfrefutation
    for char in range(len(problems_cnfref_val)):
        #obtain string as we parse through each character
        temp_str = temp_str + problems_cnfref_val[char]
        #If we reach a period
        if (problems_cnfref_val[char] == "."):
            #And if the period isn't apart of "f.tptp"
            if (problems_cnfref_val[char-1] != "f"):
                #We have reached at the end of a refutation step. Append it to temp_cnfref.
                temp_cnfref.append(temp_str)
                #print(temp_str)
                temp_str = "" #Reset string
    cnfrefutation.append(temp_cnfref) #Append temp_cnfref to cnfrefutation. 

<h3>Update the "problems" dictionary accordingly</h3>
Because every individual refutation step (respective to a given problem) has been stored in cnfrefutation as a 2D array, we will want to obtain every yth array and put it into a dictionary. Particularly, key values that belong to the proof step's respective problem's cnfrefutation.

In [203]:
for x in range(len(cnfrefutation)):
    temp_dict = {}
    for y in range(len(cnfrefutation[x])):
        #print(cnfrefutation[x][y])
        #Adds each refutation step for a respective problem into a temp dictionary.
        temp_dict.update({f"refutation step #{y}" : cnfrefutation[x][y]})
    #print(json.dumps(temp_dict, indent = 4, sort_keys = False)) #Used for debugging. Displays that temp_dict holds respective steps for a problem.
    problems[f"problem #{x}"]["cnfrefutation"] = temp_dict

<h3>Visualize where we currently are</h3>

In [204]:
print(json.dumps(problems, indent = 4, sort_keys = False)) #Prints out the respective theorem's information.

{
    "problem #0": {
        "axiom": "axio(b)&(~b|~c)",
        "conjecture": "conj(~a|b)&(b|c)",
        "ac axiom": {
            "ac axiom #0": "#cnf(i_0_2, plain, (b)).",
            "ac axiom #1": "#cnf(i_0_6, negated_conjecture, (a))."
        },
        "cnfrefutation": {
            "refutation step #0": "fof('conj(~a|b)&(b|c)', conjecture, ((~(a)|b)&(b|c)), file('f.tptp', 'conj(~a|b)&(b|c)')).",
            "refutation step #1": "fof('axio(b)&(~b|~c)', axiom, (b&(~(b)|~(c))), file('f.tptp', 'axio(b)&(~b|~c)')).",
            "refutation step #2": "fof(c_0_2, negated_conjecture, ~(((~a|b)&(b|c))), inference(fof_simplification,[status(thm)],[inference(assume_negation,[status(cth)],['conj(~a|b)&(b|c)'])])).",
            "refutation step #3": "fof(c_0_3, negated_conjecture, (((~b|a)&(~c|a))&((~b|~b)&(~c|~b))), inference(distribute,[status(thm)],[inference(fof_nnf,[status(thm)],[inference(fof_nnf,[status(thm)],[c_0_2])])])).",
            "refutation step #4": "fof(c_0_4, plain,

<h2>Parsing AC Axioms</h2>

In [205]:
#For every problem
for x in range(len(problems)):
    #print(f"problem #{x}")
    total_ac_axioms = len(problems.get(f"problem #{x}").get("ac axiom")) #Find how many ac axiom steps there are in a given problem.
    
    #For every total ac axiom step in the xth problem
    for y in range(total_ac_axioms):
        temp_dict = {}
        #obtain the value from the given key "ac axiom step #{y}"
        current_ref = problems.get(f"problem #{x}").get("ac axiom").get(f"ac axiom #{y}")
        #print(" ",f"ac axiom #{y}",current_ref)
        first_comma_index = current_ref.find(",") #takes place after the first parameter (to the left of the first comma)
        second_comma_index = current_ref.find(",", first_comma_index+1) #+1 to escape where the first index is. Otherwise it'd just stay at where first_comma_index is located.
        third_comma_index = current_ref.find(",", second_comma_index+1)
        
        
        #print("  Language:",current_ref[0:3]) #language | grabs language of the xth theorem's yth proof step.
        temp_dict.update({"language":current_ref[1:4]})
        #print("  Name:",current_ref[4:first_comma_index]) #name
        temp_dict.update({"name":current_ref[5:first_comma_index]})
        #print("  Type:",current_ref[first_comma_index+2:second_comma_index]) #type | +2 explanation: +1 accounts for the location of the comma that would be printed. Another +2 accounts for the a space after the comma.=
        temp_dict.update({"type":current_ref[first_comma_index+2:second_comma_index]})
        #print("  Logical Formula:",current_ref[second_comma_index+2:third_comma_index]) #logical formula
        temp_dict.update({"logical formula":current_ref[second_comma_index+2:third_comma_index-1]})

        #If a file with "file('f.tptp)" is NOT read then grab the src
        if "file('f.tptp'," not in current_ref:
            end_of_src_index = third_comma_index
            parentheses_checker = 0

            #Obtain optional fifth parameter (which indicates the proof status)
            #indicates that there is a fifth parameter, aka the proof status. 
            if current_ref[end_of_src_index] == ",":
                #print("    Optional Useful Parameter:",current_ref[end_of_src_index+2:len(current_ref)-2])
                temp_dict.update({"optional useful parameter":current_ref[end_of_src_index+2:len(current_ref)-2]})
            #else: #May not need this.
                #("    Optional Useful Parameter: None")
        
        #print("")
        problems[f"problem #{x}"]["ac axiom"][f"ac axiom #{y}"] = temp_dict #Update the dictionary with the new information.
        #print(temp_dict)
        #Parse through each ac axiom step. 
    #print("\n")

<h2>Parsing Refutation Steps</h2>

<h3>obtaining information about each theorem's proof step</h3>

Now that we have refutation steps stored, we need to break them down further.

We obtain the following from each refutation step:
- Language
- Name- Type
- Logical Formula
- Source (if any)
- Optional Useful Parameter (if any)

In [206]:
#For every problem
for x in range(len(problems)):
    #print(f"problem #{x}")
    #Find how many refutation steps there are in a given problem.
    total_ref_steps = len(problems.get(f"problem #{x}").get("cnfrefutation"))
    
    #For every total refutation step in the xth problem
    for y in range(total_ref_steps):
        temp_dict = {}
        #obtain the value from the given key "refutation step #{y}"
        current_ref = problems.get(f"problem #{x}").get("cnfrefutation").get(f"refutation step #{y}")
        #print(" ",f"refutation step #{y}",current_ref)
        first_comma_index = current_ref.find(",") #takes place after the first parameter (to the left of the first comma)
        second_comma_index = current_ref.find(",", first_comma_index+1) #+1 to escape where the first index is. Otherwise it'd just stay at where first_comma_index is located.
        third_comma_index = current_ref.find(",", second_comma_index+1)
        
        
        #print("  Language:",current_ref[0:3]) #language | grabs language of the xth theorem's yth proof step.
        temp_dict.update({"language":current_ref[0:3]})
        #print("  Name:",current_ref[4:first_comma_index]) #name
        temp_dict.update({"name":current_ref[4:first_comma_index]})
        #print("  Type:",current_ref[first_comma_index+2:second_comma_index]) #type | +2 explanation: +1 accounts for the location of the comma that would be printed. Another +2 accounts for the a space after the comma.=
        temp_dict.update({"type":current_ref[first_comma_index+2:second_comma_index]})
        #print("  Logical Formula:",current_ref[second_comma_index+2:third_comma_index]) #logical formula
        temp_dict.update({"logical formula":current_ref[second_comma_index+2:third_comma_index]})

        #If a file with "file('f.tptp)" is NOT read then grab the src
        if "file('f.tptp'," not in current_ref:
            end_of_src_index = third_comma_index
            parentheses_checker = 0

            #for every number in the range of where the third comma starts all the way up to the end of the string. 
            for z in range(third_comma_index, len(current_ref), 1):
                if current_ref[z] == "(":
                    parentheses_checker += 1
                elif current_ref[z] == ")":
                    parentheses_checker -= 1
                    #If we have reached the end of the fourth parameter. 
                    if parentheses_checker == 0:
                        end_of_src_index += 1
                        break
                end_of_src_index += 1
            #print("    Source:",current_ref[third_comma_index+2:end_of_src_index])
            temp_dict.update({"source":current_ref[third_comma_index+2:end_of_src_index]})

            #Obtain optional fifth parameter (which indicates the proof status)
            #indicates that there is a fifth parameter, aka the proof status. 
            if current_ref[end_of_src_index] == ",":
                #print("    Optional Useful Parameter:",current_ref[end_of_src_index+2:len(current_ref)-2])
                temp_dict.update({"optional useful parameter":current_ref[end_of_src_index+2:len(current_ref)-2]})
            #else: #May not need this.
                #("    Optional Useful Parameter: None")
        
        #print("")
        problems[f"problem #{x}"]["cnfrefutation"][f"refutation step #{y}"] = temp_dict #Update the dictionary with the new information.

        #Parse through each refutation step. 
    #print("\n")

<h3> Visualize where we are. </h3>

In [207]:
#Pretty print problems
print(json.dumps(problems, indent = 4, sort_keys = False)) #Prints out the respective theorem's information.

{
    "problem #0": {
        "axiom": "axio(b)&(~b|~c)",
        "conjecture": "conj(~a|b)&(b|c)",
        "ac axiom": {
            "ac axiom #0": {
                "language": "cnf",
                "name": "i_0_2",
                "type": "plain",
                "logical formula": "(b)"
            },
            "ac axiom #1": {
                "language": "cnf",
                "name": "i_0_6",
                "type": "negated_conjecture",
                "logical formula": "(a)"
            }
        },
        "cnfrefutation": {
            "refutation step #0": {
                "language": "fof",
                "name": "'conj(~a|b)&(b|c)'",
                "type": "conjecture",
                "logical formula": "((~(a)|b)&(b|c))"
            },
            "refutation step #1": {
                "language": "fof",
                "name": "'axio(b)&(~b|~c)'",
                "type": "axiom",
                "logical formula": "(b&(~(b)|~(c)))"
            },
            "re

<h2>Parsing "source" of Each Refutation Step to Find Number of Inferences</h2>

This allows us to indicate whether a "source" within a refutation step contains any nested inferences.

In [208]:
#For every problem
for x in range(len(problems)):
    #print(f"problem #{x}")
    #Find how many refutation steps there are in a given problem.
    total_ref_steps = len(problems.get(f"problem #{x}").get("cnfrefutation"))
    #For every total refutation step in the xth problem
    for y in range(total_ref_steps):
        temp_dict = {}
        count = 0
        #If the current refutation step has a source. (indiaates that it has inferences)
        #We also use this to check for nested inferences.
        if problems.get(f"problem #{x}").get("cnfrefutation").get(f"refutation step #{y}").get("source") != None:
            #print("Source:", problems.get(f"problem #{x}").get("cnfrefutation").get(f"refutation step #{y}").get("source"))
            current_src = problems.get(f"problem #{x}").get("cnfrefutation").get(f"refutation step #{y}").get("source")

            #Check if there are nested inferences.
            for char in re.finditer(r"inference\(", current_src):
                start = char.start() #Obtain index of beginning of "inference("
                end = char.end()  #Obtainending index of "inference("
                parentheses_tracker = 1 #Helps us to keep track of whether we have reached a closed parentheses such as in inference().

                #While we have not reached the end of src
                while end < len(current_src):
                    if current_src[end] == "(":
                        parentheses_tracker += 1
                    elif current_src[end] == ")":
                        parentheses_tracker -= 1
                        #If we reached at the end of the parentheses for the parent inference.
                        if parentheses_tracker == 0:
                            #print(current_src[start:end+1])
                            temp_dict.update({f"inference #{count}":current_src[start:end+1]})
                            count += 1
                            break
                    end += 1
            #print(temp_dict)
            #print("")
            #print(problems.get(f"problem #{x}").get("cnfrefutation").get(f"refutation step #{y}").get("source"))
            problems[f"problem #{x}"]["cnfrefutation"][f"refutation step #{y}"]["source"] = temp_dict #Update the dictionary with the new information.


    #pretty print temp_dict
    #print(json.dumps(temp_dict, indent = 4, sort_keys = False)) #Prints out the respective theorem's information.
    #problems[f"problem #{x}"]["cnfrefutation"][f"refutation step #{y}"]["source"] = temp_dict #Update the dictionary with the new information.
    #problems.get(f"problem #{x}").get("cnfrefutation").get(f"refutation step #{y}").get("source")
    
            

<h3> View where we are </h3>

In [209]:
#Pretty print problems
print(json.dumps(problems, indent = 4, sort_keys = False)) #Prints out the respective theorem's information.

{
    "problem #0": {
        "axiom": "axio(b)&(~b|~c)",
        "conjecture": "conj(~a|b)&(b|c)",
        "ac axiom": {
            "ac axiom #0": {
                "language": "cnf",
                "name": "i_0_2",
                "type": "plain",
                "logical formula": "(b)"
            },
            "ac axiom #1": {
                "language": "cnf",
                "name": "i_0_6",
                "type": "negated_conjecture",
                "logical formula": "(a)"
            }
        },
        "cnfrefutation": {
            "refutation step #0": {
                "language": "fof",
                "name": "'conj(~a|b)&(b|c)'",
                "type": "conjecture",
                "logical formula": "((~(a)|b)&(b|c))"
            },
            "refutation step #1": {
                "language": "fof",
                "name": "'axio(b)&(~b|~c)'",
                "type": "axiom",
                "logical formula": "(b&(~(b)|~(c)))"
            },
            "re

<h2>Parsing Inferences within the "source" of Each Refutation Step</h2>

Here we obtain information about each inference that is stored within the "source" of each refutation step.

We obtain the following:
- Inference Rule (such as "sr, rw, pm, ...")
- Useful Information (such as "[status(thm)]")
- Name of the premises (such as "c_0_9, c_0_10, ...")

In [210]:
#For every problem
for x in range(len(problems)):
    #print(f"problem #{x}")
    #Find how many refutation steps there are in a given problem.
    total_ref_steps = len(problems.get(f"problem #{x}").get("cnfrefutation"))
    #For every total refutation step in the xth problem
    for y in range(total_ref_steps):  
        current_src = problems.get(f"problem #{x}").get("cnfrefutation").get(f"refutation step #{y}").get("source")
        #print("===current_src===\n" + str(json.dumps(current_src, indent = 4, sort_keys = False))) #Prints out the respective theorem's information.
        
        #This only checks for refutation steps that contain a src. Some may not have a src.
        if current_src != None:
            total_inferences = len(current_src)

            #for every total inferences in the "src" of the current refutation step
            for z in range(total_inferences):
                temp_dict = {}      
                current_inference = current_src.get(f"inference #{z}")
                #temp_update.update({"inference": FILL IN HERE})
                #print(str(current_inference))
                first_comma_index = current_inference.find(",") #takes place after the first parameter (to the left of the first comma)
                second_comma_index = current_inference.find(",", first_comma_index+1) #+1 to escape where the first index is. Otherwise it'd just stay at where first_comma_index is located.
                
                temp_dict.update({"name":current_inference[10:first_comma_index]})
                temp_dict.update({"type":current_inference[first_comma_index+2:second_comma_index-1]})
                
                bracket_tracker = 1 #Helps us to keep track of whether we have reached a closed parentheses such as in inference().

                end = second_comma_index + 1 #+1 to escape the comma that would be printed. Otherwise it would just stay at where the second_comma_index is located.

                #Grabs the "name of premises" by starting out right after the second comma and ending at the end of the respective "name of premises"
                #This can usually be indicated via closed brackets. 
                temp_dict.update({"name of premises":current_inference[end+1:len(current_inference)-2]})
                #print("testing",current_inference[end+1:len(current_inference)-2])
                #print("   ", json.dumps(temp_dict, indent = 4, sort_keys = False)) #Prints out the respective theorem's information.
                problems[f"problem #{x}"]["cnfrefutation"][f"refutation step #{y}"]["source"][f"inference #{z}"] = temp_dict
    #print("\n=====new problem============")

# Final View Output

In [211]:
#Pretty print problems
print(json.dumps(problems, indent = 4, sort_keys = False)) #Prints out the respective theorem's information.

{
    "problem #0": {
        "axiom": "axio(b)&(~b|~c)",
        "conjecture": "conj(~a|b)&(b|c)",
        "ac axiom": {
            "ac axiom #0": {
                "language": "cnf",
                "name": "i_0_2",
                "type": "plain",
                "logical formula": "(b)"
            },
            "ac axiom #1": {
                "language": "cnf",
                "name": "i_0_6",
                "type": "negated_conjecture",
                "logical formula": "(a)"
            }
        },
        "cnfrefutation": {
            "refutation step #0": {
                "language": "fof",
                "name": "'conj(~a|b)&(b|c)'",
                "type": "conjecture",
                "logical formula": "((~(a)|b)&(b|c))"
            },
            "refutation step #1": {
                "language": "fof",
                "name": "'axio(b)&(~b|~c)'",
                "type": "axiom",
                "logical formula": "(b&(~(b)|~(c)))"
            },
            "re

<h2>Save parsed data to a .txt file</h2>

In [212]:
with open(parsed_file_to_txt, "w") as f:
    f.write(json.dumps(problems, indent = 4, sort_keys = False))

with open(parsed_file_to_json, "w") as f:
    f.write(json.dumps(problems))